# Kaggle Native T4 Test Repro (trainzeh6.1)

Notebook ini menjalankan semuanya lewat notebook:
1. Resolve dataset dari `/kaggle/input` lalu salin ke `/kaggle/temp`
2. Clone repo ke `/kaggle/temp`
3. Setup `.venv` dan install dependency profile
4. Download parent teacher model dari Hugging Face
5. Gabung metadata.csv + metadata_indsp.csv
6. Prepare dataset Arrow
7. Setup W&B login
8. Distillation + full training dengan preset uji T4
9. Push final checkpoint ke Hugging Face Hub (repo baru, private)
10. Simpan output inference test ke `/kaggle/working`

Semua command Python dijalankan via: `uv run --python .venv/bin/python`

Credential seperti W&B dan Hugging Face bisa diisi langsung di cell config pertama.


In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path


def resolve_secret(name: str, hardcoded_value: str = "", env_var: str | None = None) -> str:
    value = (hardcoded_value or "").strip()
    if value:
        return value
    value = os.environ.get(env_var or name, "").strip()
    if value:
        return value
    raise ValueError(
        f"{name} belum diisi. Isi langsung di cell ini atau set environment variable {env_var or name}."
    )


# ================= User Config =================
REPO_URL = "https://github.com/AneKazek/malesbgt.git"
REPO_BRANCH = "bokepls"

# ================= API Keys / Tokens =================
# Isi langsung string di bawah ini kalau mau hardcode di notebook.
# Kalau dikosongkan, notebook akan fallback ke environment variable yang namanya sama.
WANDB_API_KEY_RAW = ""
HF_TOKEN_RAW = ""

WANDB_API_KEY = resolve_secret("WANDB_API_KEY", WANDB_API_KEY_RAW)
HF_TOKEN = resolve_secret("HF_TOKEN", HF_TOKEN_RAW)

WANDB_ENTITY = "haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember"
WANDB_PROJECT = "kaceve"

HF_REPO_ID = "Eempostor/F5-TTS-INDO-FINETUNE-V2"
HF_CKPT_FILENAME = "f5_tts_indo_v2.pt"

# Optional output repo naming for pushed checkpoints (selalu buat repo private baru)
HF_OUTPUT_REPO_PREFIX = os.environ.get("HF_OUTPUT_REPO_PREFIX", "kcv-tts-kaggle-t4-test-ckpt")
HF_OUTPUT_REPO_OWNER = os.environ.get("HF_OUTPUT_REPO_OWNER", "")

# ================= Kaggle Native Paths =================
INPUT_ROOT = Path("/kaggle/input")
WORKDIR = Path("/kaggle/temp")
OUTPUT_DIR = Path("/kaggle/working")
INFER_OUTPUT_DIR = OUTPUT_DIR / "inference_tests"
DATASET_INPUT_DIR_RAW = os.environ.get("DATASET_INPUT_DIR", "/kaggle/input/tts-indo/data")
USE_METADATA_INDSP = False

DATASET_ROOT = WORKDIR / "datasets" / "tts_indo"
DATASET_DATA_DIR = DATASET_ROOT / "data"
CSV_1 = DATASET_DATA_DIR / "metadata.csv"
CSV_2 = (DATASET_DATA_DIR / "metadata_indsp.csv") if USE_METADATA_INDSP else None

REPO_DIR = WORKDIR / "kcv-tts"
VENV_DIR = REPO_DIR / ".venv"
VENV_PY = VENV_DIR / "bin/python"

HF_OUT_DIR = REPO_DIR / "ckpts/hf/Eempostor_F5-TTS-INDO-FINETUNE-V2"
TEACHER_CKPT = HF_OUT_DIR / HF_CKPT_FILENAME
MERGED_CSV = REPO_DIR / "data/metadata_merged.csv"
PREPARED_DATASET_DIR = REPO_DIR / "data/datasetku_pinyin"

# T4-oriented test defaults
H100_MIXED_PRECISION = "fp16"
H100_DISTILL_BATCH_FRAMES = 9600
H100_FULL_BATCH_FRAMES = 9600
H100_MAX_SAMPLES = 32
H100_NUM_WORKERS = int(os.environ.get("TRAIN_NUM_WORKERS", "4"))
H100_USE_FLASH_ATTN = os.environ.get("H100_USE_FLASH_ATTN", "0") == "1"


def run_cmd(cmd, cwd=None, env=None, timeout=None):
    printable = cmd if isinstance(cmd, str) else " ".join(str(x) for x in cmd)
    print("\n$", printable)
    if timeout is not None:
        print(f"(timeout={timeout}s)")
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        timeout=timeout,
    )


def run_py(args, cwd=None, env=None, timeout=None):
    return run_cmd(["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", *args], cwd=cwd, env=env, timeout=timeout)


WORKDIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INFER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Config siap.")
print("INPUT_ROOT:", INPUT_ROOT)
print("DATASET_INPUT_DIR_RAW:", DATASET_INPUT_DIR_RAW)
print("WORKDIR:", WORKDIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Credential source:", {
    "wandb_api_key": "hardcoded" if WANDB_API_KEY_RAW else "env",
    "hf_token": "hardcoded" if HF_TOKEN_RAW else "env",
})
print("T4 test preset:", {
    "mixed_precision": H100_MIXED_PRECISION,
    "distill_batch_frames": H100_DISTILL_BATCH_FRAMES,
    "full_batch_frames": H100_FULL_BATCH_FRAMES,
    "max_samples": H100_MAX_SAMPLES,
    "num_workers": H100_NUM_WORKERS,
    "use_flash_attn": H100_USE_FLASH_ATTN,
})


In [ ]:
# 1) Resolve dataset dari /kaggle/input, copy ke /kaggle/temp, lalu clone repo
candidate_csvs = []

preferred_csv = Path(DATASET_INPUT_DIR_RAW) / "metadata.csv"
candidate_csvs.append(preferred_csv)
candidate_csvs.extend(sorted(INPUT_ROOT.glob("**/metadata.csv")))

source_csv1 = next((p for p in candidate_csvs if p.exists()), None)
if source_csv1 is None:
    raise FileNotFoundError(
        f"metadata.csv tidak ditemukan. Cek mount dataset Kaggle atau set DATASET_INPUT_DIR. Tried: {preferred_csv}"
    )

source_data_dir = source_csv1.parent
source_dataset_root = source_data_dir.parent
source_csv2 = source_data_dir / "metadata_indsp.csv"

print("Mounted dataset source:", source_dataset_root)
print("Mounted CSV_1:", source_csv1)
print("Mounted CSV_2:", source_csv2 if source_csv2.exists() else None)

if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)
DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(source_dataset_root, DATASET_ROOT)

DATASET_DATA_DIR = DATASET_ROOT / source_data_dir.name
CSV_1 = DATASET_DATA_DIR / "metadata.csv"
CSV_2 = (DATASET_DATA_DIR / "metadata_indsp.csv") if USE_METADATA_INDSP and (DATASET_DATA_DIR / "metadata_indsp.csv").exists() else None

print("Copied dataset root:", DATASET_ROOT)
print("Working CSV_1:", CSV_1)
print("Working CSV_2:", CSV_2)
run_cmd(["ls", "-lah", str(DATASET_ROOT)])

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run_cmd(["git", "clone", "--recursive", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])
run_cmd(["ls", "-lah", str(REPO_DIR)])


In [ ]:
# 2) Setup .venv + install dependency profile (local/kaggle parity)
if shutil.which("uv") is None:
    run_cmd(["python3", "-m", "pip", "install", "-U", "uv"])

TARGET_PY_MM = "3.11"
TARGET_PY = Path(f"/usr/bin/python{TARGET_PY_MM}")

run_cmd(["apt-get", "update", "-y"])
run_cmd([
    "apt-get",
    "install",
    "-y",
    f"python{TARGET_PY_MM}",
    f"python{TARGET_PY_MM}-venv",
    f"python{TARGET_PY_MM}-dev",
    "build-essential",
])

if not TARGET_PY.exists():
    raise FileNotFoundError(f"Interpreter target tidak ditemukan: {TARGET_PY}")

run_cmd(["uv", "venv", "--python", str(TARGET_PY), "--clear", str(VENV_DIR)])
run_cmd([str(VENV_PY), "-c", "import sys; print('venv python =', sys.version); assert sys.version_info[:2] == (3, 11)"])

# Setuptools<82 menghindari masalah build extension tertentu (mamba/causal-conv1d).
run_cmd([
    "uv",
    "pip",
    "install",
    "--python",
    str(VENV_PY),
    "--upgrade",
    "pip",
    "wheel",
    "setuptools<82",
])

req_candidates = [
    REPO_DIR / "requirements-torch28-cu12-localmatch.txt",
    REPO_DIR / "requirements-kaggle-torch210.txt",
]

REQ_PROFILE = next((p for p in req_candidates if p.exists()), None)
if REQ_PROFILE is None:
    raise FileNotFoundError("Tidak menemukan file profile requirements untuk setup training.")

print("Using dependency profile:", REQ_PROFILE)
run_cmd([
    "uv",
    "pip",
    "install",
    "--python",
    str(VENV_PY),
    "--index-strategy",
    "unsafe-best-match",
    "-r",
    str(REQ_PROFILE),
])

# Final guard: pastikan stack torch sesuai target training mamba.
run_cmd([
    "uv",
    "pip",
    "install",
    "--python",
    str(VENV_PY),
    "--index-url",
    "https://download.pytorch.org/whl/cu128",
    "--extra-index-url",
    "https://pypi.org/simple",
    "--index-strategy",
    "unsafe-best-match",
    "--force-reinstall",
    "--no-cache-dir",
    "torch==2.8.0+cu128",
    "torchvision==0.23.0+cu128",
    "torchaudio==2.8.0+cu128",
    "nvidia-nccl-cu12==2.27.3",
    "nvidia-nvjitlink-cu12==12.8.93",
])

run_cmd([
    str(VENV_PY),
    "-c",
    "import torch, torchvision, torchaudio, setuptools; "
    "print('torch =', torch.__version__, 'cuda =', torch.version.cuda); "
    "print('torchvision =', torchvision.__version__); "
    "print('torchaudio =', torchaudio.__version__); "
    "print('setuptools =', setuptools.__version__)"
])

In [ ]:
# 3) Validasi GPU untuk preset uji T4
run_py([
    "-c",
    "import torch; "
    "print('torch', torch.__version__); "
    "print('cuda_count', torch.cuda.device_count()); "
    "[print(i, torch.cuda.get_device_name(i), 'cc', torch.cuda.get_device_capability(i)) for i in range(torch.cuda.device_count())]; "
    "assert torch.cuda.device_count() >= 1, 'GPU tidak terdeteksi'",
])

In [ ]:
# 4) Download parent teacher model dari HF
run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "-U", "huggingface_hub"])

download_script = "\n".join([
    "from pathlib import Path",
    "from huggingface_hub import hf_hub_download",
    f"repo_id = {HF_REPO_ID!r}",
    f"filename = {HF_CKPT_FILENAME!r}",
    f"out_dir = Path(r'{HF_OUT_DIR}')",
    "out_dir.mkdir(parents=True, exist_ok=True)",
    "path = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(out_dir), local_dir_use_symlinks=False)",
    "print(path)",
])

run_py(["-c", download_script], cwd=REPO_DIR)
run_cmd(["ls", "-lah", str(HF_OUT_DIR)])

In [ ]:
# 5) Merge 2 CSV metadata -> audio_file|text (pipe-delimited)
import csv
import pandas as pd

def _normalize_metadata_df(df: pd.DataFrame, path: Path) -> pd.DataFrame:
    df.columns = [str(c).strip() for c in df.columns]

    audio_candidates = ["audio_file", "audio_path", "wav_path", "path", "file"]
    text_candidates = ["text", "transcript", "sentence", "normalized_text", "utterance"]

    audio_col = next((c for c in audio_candidates if c in df.columns), None)
    text_col = next((c for c in text_candidates if c in df.columns), None)

    if audio_col is None or text_col is None:
        raise ValueError(f"Kolom tidak cocok di {path}. Dapat: {list(df.columns)}")

    out = df[[audio_col, text_col]].copy()
    out.columns = ["audio_file", "text"]
    out["audio_file"] = out["audio_file"].astype(str).str.strip()
    out["text"] = out["text"].astype(str).str.strip()
    out = out[(out["audio_file"] != "") & (out["text"] != "")]

    source_dir = path.parent
    prefer_indsp = "indsp" in path.name.lower()

    def absolutize(p: str) -> str:
        raw = str(p).strip().strip('"').strip("'")
        pp = Path(raw).expanduser()
        if pp.is_absolute():
            return str(pp)

        has_dir = ("/" in raw) or ("\\" in raw)
        candidates = [source_dir / pp]

        # metadata_indsp sering berisi filename polos (tanpa folder), jadi coba prefix indsp/.
        if not has_dir:
            if prefer_indsp:
                candidates.insert(0, source_dir / "indsp" / pp)
            else:
                candidates.append(source_dir / "wavs" / pp)
                candidates.append(source_dir / "indsp" / pp)

        for cand in candidates:
            if cand.exists():
                return str(cand.resolve())

        return str(candidates[0].resolve())

    out["audio_file"] = out["audio_file"].map(absolutize)
    out["source_csv"] = str(path)
    return out

def _manual_parse_metadata(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8-sig", errors="replace") as f:
        first_non_empty = ""
        for ln in f:
            if ln.strip():
                first_non_empty = ln
                break
        f.seek(0)

        delim = "|" if first_non_empty.count("|") >= first_non_empty.count(",") else ","

        for i, ln in enumerate(f):
            ln = ln.strip()
            if not ln:
                continue
            if i == 0 and "audio_file" in ln.lower() and "text" in ln.lower():
                continue
            if delim not in ln:
                continue
            audio, text = ln.split(delim, 1)
            audio = audio.strip().strip('"')
            text = text.strip()
            if audio and text:
                rows.append((audio, text))

    df = pd.DataFrame(rows, columns=["audio_file", "text"])
    df["source_csv"] = str(path)
    return df

def load_metadata(path: Path | None) -> pd.DataFrame:
    if path is None:
        print("load_metadata: path None, skip.")
        return pd.DataFrame(columns=["audio_file", "text", "source_csv"])
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Metadata CSV tidak ditemukan: {path}")

    parse_attempts = [
        dict(
            sep="|",
            engine="python",
            dtype=str,
            keep_default_na=False,
            encoding="utf-8-sig",
            on_bad_lines="skip",
            quoting=csv.QUOTE_NONE,
        ),
        dict(
            sep=",",
            engine="python",
            dtype=str,
            keep_default_na=False,
            encoding="utf-8-sig",
            on_bad_lines="skip",
            quoting=csv.QUOTE_NONE,
        ),
    ]

    for kwargs in parse_attempts:
        try:
            df = pd.read_csv(path, **kwargs)
            out = _normalize_metadata_df(df, path)
            if len(out) > 0:
                return out
        except Exception:
            pass

    print(f"Parser fallback aktif untuk {path}")
    return _normalize_metadata_df(_manual_parse_metadata(path), path)

sources = [CSV_1]
if CSV_2 is not None:
    sources.append(CSV_2)
else:
    print("CSV_2/metadata_indsp dimatikan sementara. Hanya pakai CSV_1.")

dfs = [load_metadata(path) for path in sources]
merged = pd.concat(dfs, ignore_index=True).drop_duplicates(subset=["audio_file", "text"])

exists_mask = merged["audio_file"].map(lambda p: Path(p).exists())
missing = int((~exists_mask).sum())
if missing:
    print(f"Dropping {missing} rows with missing audio paths.")
merged = merged[exists_mask].copy()

MERGED_CSV.parent.mkdir(parents=True, exist_ok=True)
merged[["audio_file", "text"]].to_csv(MERGED_CSV, sep="|", index=False)

print("Merged rows (after drop missing):", len(merged))
print("Saved:", MERGED_CSV)
print("Source CSVs:", *sources)
print(merged[["audio_file", "text"]].head(5))

In [ ]:
# 6) Siapkan data/Emilia_ZH_EN_pinyin/vocab.txt (minimal, hanya yang dibutuhkan prepare_csv_wavs.py)
EMILIA_VOCAB_DIR = REPO_DIR / "data" / "Emilia_ZH_EN_pinyin"
EMILIA_VOCAB_PATH = EMILIA_VOCAB_DIR / "vocab.txt"
EMILIA_VOCAB_DIR.mkdir(parents=True, exist_ok=True)

if EMILIA_VOCAB_PATH.exists() and EMILIA_VOCAB_PATH.stat().st_size > 0:
    print("Pretrained vocab sudah ada:", EMILIA_VOCAB_PATH)
else:
    run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "-U", "huggingface_hub"])

    vocab_fetch_script = "\n".join([
        "from pathlib import Path",
        "import shutil",
        "from huggingface_hub import hf_hub_download",
        f"target = Path(r'{EMILIA_VOCAB_PATH}')",
        "target.parent.mkdir(parents=True, exist_ok=True)",
        "candidates = [",
        "    ('SWivid/F5-TTS', 'F5TTS_Base/vocab.txt'),",
        "    ('SWivid/F5-TTS', 'F5TTS_v1_Base/vocab.txt'),",
        "]",
        "last_err = None",
        "for repo_id, filename in candidates:",
        "    try:",
        "        src = Path(hf_hub_download(repo_id=repo_id, filename=filename))",
        "        shutil.copy2(src, target)",
        "        print(f'Downloaded vocab from {repo_id}/{filename} -> {target}')",
        "        break",
        "    except Exception as e:",
        "        print(f'Gagal dari {repo_id}/{filename}: {e}')",
        "        last_err = e",
        "else:",
        "    raise RuntimeError(f'Gagal download vocab Emilia_ZH_EN_pinyin: {last_err}')",
        "print('vocab exists:', target.exists(), 'size:', target.stat().st_size if target.exists() else -1)",
    ])

    run_py(["-c", vocab_fetch_script], cwd=REPO_DIR)

run_cmd(["ls", "-lah", str(EMILIA_VOCAB_DIR)])

In [ ]:
# 7) Jalankan prepare_csv_wavs.py di notebook (Kaggle-native vocab detect)
PREPARED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

expected_vocab = REPO_DIR / "data" / "Emilia_ZH_EN_pinyin" / "vocab.txt"
if not expected_vocab.exists() or expected_vocab.stat().st_size == 0:
    data_root = REPO_DIR / "data"
    fallback = None
    for cand in data_root.glob("**/vocab.txt"):
        if cand.is_file() and cand.stat().st_size > 0:
            fallback = cand
            break

    if fallback is None:
        raise FileNotFoundError(
            f"vocab.txt tidak ditemukan untuk finetune prepare. Expected: {expected_vocab}"
        )

    expected_vocab.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(fallback, expected_vocab)
    print("Fallback vocab dipakai:", fallback, "->", expected_vocab)

print("Using pretrained vocab:", expected_vocab)

# Self-heal jika setup cell tidak dijalankan ulang setelah perubahan dependency profile.
try:
    run_py(["-c", "import f5_tts; print('f5_tts import ok')"], cwd=REPO_DIR)
except subprocess.CalledProcessError:
    print("f5_tts belum terinstall di venv, install editable package...")
    run_cmd([
        "uv",
        "pip",
        "install",
        "--python",
        str(VENV_PY),
        "-e",
        str(REPO_DIR),
    ], cwd=REPO_DIR)

run_py([
    "src/f5_tts/train/datasets/prepare_csv_wavs.py",
    str(MERGED_CSV),
    str(PREPARED_DATASET_DIR),
    "--workers",
    str(H100_NUM_WORKERS),
], cwd=REPO_DIR)

run_cmd(["ls", "-lah", str(PREPARED_DATASET_DIR)])


In [ ]:
# 8) W&B hardcoded login + sanity logging
env = os.environ.copy()
env["WANDB_API_KEY"] = WANDB_API_KEY
env["WANDB_ENTITY"] = WANDB_ENTITY
env["WANDB_PROJECT"] = WANDB_PROJECT

wandb_smoke = """
import os
import random
import wandb

api_key = os.environ["WANDB_API_KEY"]
entity = os.environ["WANDB_ENTITY"]
project = os.environ.get("WANDB_PROJECT", "kaceve")

wandb.login(key=api_key)
run = wandb.init(
    entity=entity,
    project=project,
    config={
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)
epochs = 1
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset
    run.log({"acc": acc, "loss": loss})
run.finish()
print("wandb sanity done")
""".strip()

run_py(["-c", wandb_smoke], cwd=REPO_DIR, env=env)

In [ ]:
# 9) Persiapan runtime training + mamba (Kaggle-native, tanpa menjalankan training)
env = os.environ.copy()
env["WANDB_API_KEY"] = WANDB_API_KEY
env["WANDB_ENTITY"] = WANDB_ENTITY
env["WANDB_PROJECT"] = WANDB_PROJECT

CKPT_ROOT_WORKING = WORKDIR / "ckpts"
DISTILL_TAG = "distill_final_datasetku"
FULL_TAG = "full_final_datasetku"

distill_dir_abs = CKPT_ROOT_WORKING / DISTILL_TAG
full_dir_abs = CKPT_ROOT_WORKING / FULL_TAG
distill_dir_abs.mkdir(parents=True, exist_ok=True)
full_dir_abs.mkdir(parents=True, exist_ok=True)

# train.py menyimpan checkpoint ke repo_root/ckpts.save_dir, jadi dari REPO_DIR cukup naik 1 level ke WORKDIR.
distill_save_dir_rel = f"../ckpts/{DISTILL_TAG}"
full_save_dir_rel = f"../ckpts/{FULL_TAG}"

no_periodic_ckpt_overrides = [
    "ckpts.save_per_updates=999999999",
    "ckpts.last_per_updates=999999999",
    "ckpts.keep_last_n_checkpoints=0",
    "ckpts.log_samples=False",
]

MAMBA_PROBE_TIMEOUT_SEC = 90
MAMBA_POST_REPAIR_TIMEOUT_SEC = 240


def _build_runtime_env(base_env):
    runtime_env = base_env.copy()
    site_pkgs_candidates = sorted((VENV_DIR / "lib").glob("python*/site-packages"))
    if not site_pkgs_candidates:
        return runtime_env

    sp = site_pkgs_candidates[-1]
    cuda_lib_rels = [
        "nvidia/cublas/lib",
        "nvidia/cuda_runtime/lib",
        "nvidia/cudnn/lib",
        "nvidia/cufft/lib",
        "nvidia/curand/lib",
        "nvidia/cusolver/lib",
        "nvidia/cusparse/lib",
        "nvidia/nccl/lib",
        "nvidia/nvjitlink/lib",
    ]
    cuda_libs = [str(sp / rel) for rel in cuda_lib_rels if (sp / rel).exists()]
    if cuda_libs:
        current = runtime_env.get("LD_LIBRARY_PATH", "")
        runtime_env["LD_LIBRARY_PATH"] = ":".join(cuda_libs + ([current] if current else []))
        print("LD_LIBRARY_PATH prepared for CUDA libs in venv.")

    runtime_env.setdefault("PYTHONFAULTHANDLER", "1")
    runtime_env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    return runtime_env


runtime_env = _build_runtime_env(env)


def run_py_nosync(args, cwd=None, timeout=None):
    return run_cmd(
        ["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", *args],
        cwd=cwd,
        env=runtime_env,
        timeout=timeout,
    )


def _probe_mamba(timeout_sec=MAMBA_PROBE_TIMEOUT_SEC) -> None:
    probe_script = "\n".join([
        "print('probe: import torch...', flush=True)",
        "import torch",
        "print('probe: torch =', torch.__version__, 'cuda =', torch.version.cuda, flush=True)",
        "print('probe: import mamba_ssm...', flush=True)",
        "import mamba_ssm",
        "print('probe: import selective_scan_cuda...', flush=True)",
        "import selective_scan_cuda",
        "print('mamba probe ok', flush=True)",
    ])
    run_py_nosync(["-u", "-X", "faulthandler", "-c", probe_script], cwd=REPO_DIR, timeout=timeout_sec)


def _ensure_torch_compat() -> None:
    check_script = "\n".join([
        "import torch",
        "print('torch =', torch.__version__, 'cuda =', torch.version.cuda)",
        "ok = torch.__version__.startswith('2.8.0') and str(torch.version.cuda).startswith('12.8')",
        "raise SystemExit(0 if ok else 1)",
    ])
    try:
        run_py_nosync(["-c", check_script], cwd=REPO_DIR, timeout=60)
        return
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("Torch stack tidak cocok, paksa reinstall ke torch 2.8.0+cu128.")

    run_cmd([
        "uv",
        "pip",
        "install",
        "--python",
        str(VENV_PY),
        "--index-url",
        "https://download.pytorch.org/whl/cu128",
        "--extra-index-url",
        "https://pypi.org/simple",
        "--index-strategy",
        "unsafe-best-match",
        "--force-reinstall",
        "--no-cache-dir",
        "torch==2.8.0+cu128",
        "torchvision==0.23.0+cu128",
        "torchaudio==2.8.0+cu128",
        "nvidia-nccl-cu12==2.27.3",
        "nvidia-nvjitlink-cu12==12.8.93",
    ], cwd=REPO_DIR)

    run_py_nosync(["-c", check_script], cwd=REPO_DIR, timeout=60)


def _ensure_python_headers() -> None:
    py_mm = subprocess.check_output(
        [str(VENV_PY), "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"],
        text=True,
    ).strip()
    py_header = Path(f"/usr/include/python{py_mm}/Python.h")
    if py_header.exists():
        return

    print(f"Python headers tidak ditemukan ({py_header}), install python{py_mm}-dev...")
    run_cmd(["apt-get", "update", "-y"])
    run_cmd(["apt-get", "install", "-y", f"python{py_mm}-dev", "build-essential"])

    if not py_header.exists():
        raise FileNotFoundError(f"Python.h tetap tidak ditemukan di {py_header}")


def _repair_mamba() -> None:
    print("Repair mamba dimulai...")
    _ensure_torch_compat()
    _ensure_python_headers()

    run_cmd(
        [str(VENV_PY), "-m", "pip", "install", "--upgrade", "pip", "wheel", "ninja", "setuptools<82"],
        cwd=REPO_DIR,
        env=runtime_env,
    )

    wheel_env = runtime_env.copy()
    wheel_env["TORCH_CUDA_ARCH_LIST"] = "9.0"
    wheel_env["MAX_JOBS"] = "8"

    wheel_cmd = [
        "uv",
        "pip",
        "install",
        "--python",
        str(VENV_PY),
        "--index-url",
        "https://pypi.org/simple",
        "--extra-index-url",
        "https://download.pytorch.org/whl/cu128",
        "--index-strategy",
        "unsafe-best-match",
        "--force-reinstall",
        "--no-cache-dir",
        "--prefer-binary",
        "--no-build-isolation",
        "--no-deps",
        "causal-conv1d",
        "mamba-ssm",
    ]

    try:
        run_cmd(wheel_cmd, cwd=REPO_DIR, env=wheel_env)
        _probe_mamba(timeout_sec=120)
        print("Mamba wheel kompatibel.")
        return
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("Wheel mamba gagal/timeout, lanjut build from source...")

    build_env = wheel_env.copy()
    build_env["MAMBA_FORCE_BUILD"] = "TRUE"
    build_env["CAUSAL_CONV1D_FORCE_BUILD"] = "TRUE"

    run_cmd(
        [
            str(VENV_PY),
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--no-build-isolation",
            "--force-reinstall",
            "--no-binary",
            ":all:",
            "--no-deps",
            "causal-conv1d",
        ],
        cwd=REPO_DIR,
        env=build_env,
    )
    run_cmd(
        [
            str(VENV_PY),
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--no-build-isolation",
            "--force-reinstall",
            "--no-binary",
            ":all:",
            "--no-deps",
            "mamba-ssm",
        ],
        cwd=REPO_DIR,
        env=build_env,
    )

    _probe_mamba(timeout_sec=MAMBA_POST_REPAIR_TIMEOUT_SEC)
    print("Mamba berhasil dibangun dari source.")


try:
    _probe_mamba()
    print("mamba_mode: enabled (fast-path)")
except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
    print("Probe mamba gagal/timeout. Menjalankan repair...")
    _repair_mamba()
    _probe_mamba(timeout_sec=MAMBA_POST_REPAIR_TIMEOUT_SEC)
    print("mamba_mode: enabled (after repair)")

print("Persiapan selesai. Lanjut ke cell distill.")


In [ ]:
# flash_attn opt-in dimatikan untuk preset T4 test
H100_USE_FLASH_ATTN = False
os.environ["H100_USE_FLASH_ATTN"] = "0"
print("H100_USE_FLASH_ATTN =", H100_USE_FLASH_ATTN)
print("Skipping flash_attn install for T4 test preset.")


In [ ]:
# 9) HOTFIX runtime (sementara): patch checkpoint-compat di trainer.py
from pathlib import Path

trainer_path = REPO_DIR / "src/f5_tts/model/trainer.py"
src = trainer_path.read_text(encoding="utf-8")

if "Checkpoint loaded in compatibility mode; update reset to 0" in src:
    print("Hotfix sudah ada di trainer.py")
else:
    # Gunakan string literal dengan indent asli supaya patch aman untuk runtime process lain.
    old_ema_block = (
        "        if self.is_main:\n"
        "            self.ema_model.load_state_dict(checkpoint[\"ema_model_state_dict\"])\n"
    )
    new_ema_block = (
        "        ema_loaded_strict = True\n"
        "        if self.is_main:\n"
        "            try:\n"
        "                self.ema_model.load_state_dict(checkpoint[\"ema_model_state_dict\"])\n"
        "            except RuntimeError as exc:\n"
        "                ema_loaded_strict = False\n"
        "                print(f\"Strict EMA checkpoint load failed, retrying non-strict for compatibility: {exc}\")\n"
        "                self.ema_model.load_state_dict(checkpoint[\"ema_model_state_dict\"], strict=False)\n"
    )

    old_resume_block = (
        "            try:\n"
        "                self.accelerator.unwrap_model(self.model).load_state_dict(checkpoint[\"model_state_dict\"])\n"
        "            except RuntimeError as exc:\n"
        "                print(f\"Strict checkpoint load failed, retrying non-strict for compatibility: {exc}\")\n"
        "                self.accelerator.unwrap_model(self.model).load_state_dict(checkpoint[\"model_state_dict\"], strict=False)\n"
        "            self.optimizer.load_state_dict(checkpoint[\"optimizer_state_dict\"])\n"
        "            if self.scheduler:\n"
        "                self.scheduler.load_state_dict(checkpoint[\"scheduler_state_dict\"])\n"
        "            update = checkpoint[\"update\"]\n"
    )

    new_resume_block = (
        "            model_loaded_strict = True\n"
        "            try:\n"
        "                self.accelerator.unwrap_model(self.model).load_state_dict(checkpoint[\"model_state_dict\"])\n"
        "            except RuntimeError as exc:\n"
        "                model_loaded_strict = False\n"
        "                print(f\"Strict checkpoint load failed, retrying non-strict for compatibility: {exc}\")\n"
        "                self.accelerator.unwrap_model(self.model).load_state_dict(checkpoint[\"model_state_dict\"], strict=False)\n"
        "\n"
        "            optimizer_loaded = True\n"
        "            try:\n"
        "                self.optimizer.load_state_dict(checkpoint[\"optimizer_state_dict\"])\n"
        "            except Exception as exc:\n"
        "                optimizer_loaded = False\n"
        "                print(f\"Optimizer checkpoint load failed, using fresh optimizer state: {exc}\")\n"
        "\n"
        "            scheduler_loaded = True\n"
        "            if self.scheduler:\n"
        "                try:\n"
        "                    self.scheduler.load_state_dict(checkpoint[\"scheduler_state_dict\"])\n"
        "                except Exception as exc:\n"
        "                    scheduler_loaded = False\n"
        "                    print(f\"Scheduler checkpoint load failed, using fresh scheduler state: {exc}\")\n"
        "\n"
        "            if model_loaded_strict and optimizer_loaded and scheduler_loaded and ema_loaded_strict:\n"
        "                update = checkpoint[\"update\"]\n"
        "            else:\n"
        "                update = 0\n"
        "                print(\n"
        "                    \"Checkpoint loaded in compatibility mode; update reset to 0 \"\n"
        "                    \"to avoid resuming with incompatible optimizer/scheduler state.\"\n"
        "                )\n"
    )

    missing = []
    if old_ema_block in src:
        src = src.replace(old_ema_block, new_ema_block, 1)
    elif new_ema_block in src:
        pass
    else:
        missing.append("EMA load block")

    if old_resume_block in src:
        src = src.replace(old_resume_block, new_resume_block, 1)
    elif new_resume_block in src:
        pass
    else:
        missing.append("resume block")

    if missing:
        raise RuntimeError(
            "Gagal patch trainer.py, blok tidak ditemukan: " + ", ".join(missing)
        )

    trainer_path.write_text(src, encoding="utf-8")
    print("Hotfix berhasil diterapkan:", trainer_path)

run_py(["-m", "py_compile", "src/f5_tts/model/trainer.py"], cwd=REPO_DIR)
print("trainer.py syntax OK")
run_py([
    "-c",
    "from f5_tts.model.trainer import Trainer; print('trainer import OK after hotfix')",
], cwd=REPO_DIR)

In [ ]:
# 10) Distill phase — EarlyBiMamba warmup (T4 test preset)
import json as _json
import subprocess as _sp

# ── Probe VRAM tersedia untuk auto-tune batch size ────────────────────────────
vram_probe = _sp.run(
    ["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", "-c",
     "import torch; print(torch.cuda.get_device_properties(0).total_memory // (1024**2))"],
    cwd=str(REPO_DIR), env=runtime_env, capture_output=True, text=True,
)
try:
    _vram_mb = int(vram_probe.stdout.strip().splitlines()[-1])
except Exception:
    _vram_mb = 40960  # fallback A100
print(f"VRAM detected: {_vram_mb} MB ({_vram_mb // 1024} GB)")

# Distill = 2x forward (teacher + student), jadi pakai preset konservatif untuk T4
# Fixed preset untuk uji T4 Kaggle
_distill_batch  = 9600
_distill_maxsmp = 32
_distill_nwork  = 4

print(f"Auto-tuned distill batch : {_distill_batch} frames")
print(f"Auto-tuned max_samples   : {_distill_maxsmp}")
print(f"Auto-tuned num_workers   : {_distill_nwork}")

# ── Estimasi steps per epoch ──────────────────────────────────────────────────
estimate_script = f"""
import json, csv
from pathlib import Path

total_frames = 0
method = 'unknown'
ds_path = Path(r'{PREPARED_DATASET_DIR}')

try:
    import pyarrow as pa
    arrow_files = sorted(ds_path.glob('*.arrow'))
    if not arrow_files:
        raise FileNotFoundError('no arrow files')
    for f in arrow_files:
        reader = pa.ipc.open_file(pa.memory_map(str(f), 'r'))
        table = reader.read_all()
        if 'duration' in table.schema.names:
            for d in table['duration'].to_pylist():
                if d:
                    total_frames += int(float(d) * 24000 / 256)
    method = 'pyarrow_duration'
except Exception as e1:
    try:
        import wave
        wav_files = list(ds_path.rglob('*.wav'))
        if not wav_files:
            raise FileNotFoundError('no wav files')
        for wf in wav_files:
            try:
                with wave.open(str(wf)) as w:
                    total_frames += w.getnframes() // 256
            except Exception:
                pass
        method = 'wav_scan'
    except Exception as e2:
        with open(r'{MERGED_CSV}', newline='', encoding='utf-8') as f:
            n = sum(1 for row in csv.reader(f, delimiter='|') if len(row) >= 2)
        total_frames = n * int(5 * 24000 / 256)
        method = f'csv_estimate(n={{n}})'

steps_per_epoch = max(1, total_frames // {_distill_batch})
print(json.dumps({{"total_frames": total_frames, "steps_per_epoch": steps_per_epoch, "method": method}}))
"""

_proc = _sp.run(
    ["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", "-u", "-c", estimate_script],
    cwd=str(REPO_DIR), env=runtime_env, capture_output=True, text=True,
)
print(_proc.stdout)
if _proc.returncode != 0:
    print("STDERR:", _proc.stderr)

_json_line = next(
    (ln for ln in reversed(_proc.stdout.strip().splitlines()) if ln.strip().startswith("{")),
    None,
)
if _json_line:
    _stats = _json.loads(_json_line)
    _steps_per_epoch = _stats["steps_per_epoch"]
    print(f"Dataset stats  : {_stats}")
else:
    _steps_per_epoch = 100
    print(f"WARNING: gagal parse estimasi, pakai fallback {_steps_per_epoch} steps/epoch")

# ── Hitung epochs & warmup ────────────────────────────────────────────────────
DISTILL_EPOCHS  = 5
DISTILL_WARMUP  = max(50, min(300, int(DISTILL_EPOCHS * _steps_per_epoch * 0.10)))

print(f"steps_per_epoch : {_steps_per_epoch}")
print(f"DISTILL_EPOCHS  : {DISTILL_EPOCHS}  (total ~{DISTILL_EPOCHS * _steps_per_epoch} steps)")
print(f"DISTILL_WARMUP  : {DISTILL_WARMUP}")

# ── T4-oriented runtime env ───────────────────────────────────────────────────
h100_env = runtime_env.copy()
h100_env.update({
    # Alokator CUDA konservatif untuk T4
    "PYTORCH_CUDA_ALLOC_CONF":      "expandable_segments:True,roundup_power2_divisions:16",
    # Matikan sinkronisasi CUDA yang tidak perlu di single-GPU
    "CUDA_LAUNCH_BLOCKING":         "0",
    # Maksimalkan thread CPU untuk DataLoader
    "OMP_NUM_THREADS":              str(_distill_nwork),
    "MKL_NUM_THREADS":              str(_distill_nwork),
    # fp16 lebih cocok untuk T4
    "TORCH_ALLOW_TF32_CUBLAS_OVERRIDE": "1",
    # Aktifkan TF32 untuk matmul (gratis speedup ~2x di non-critical path)
    "TORCH_CUDNN_V8_API_ENABLED":   "1",
    # Prefetch DataLoader lebih agresif
    "PYTHONFAULTHANDLER":           "1",
})

# ── Attn backend ──────────────────────────────────────────────────────────────
attn_backend_overrides = []
if H100_USE_FLASH_ATTN:
    try:
        run_py_nosync(["-c", "import flash_attn; print('flash_attn ok')"], cwd=REPO_DIR, timeout=60)
        attn_backend_overrides = ["model.arch.attn_backend=flash_attn"]
        print("Using flash_attn backend.")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("flash_attn tidak tersedia, fallback torch.")

# ── Periodic ckpt untuk distill (berbeda dari full training) ─────────────────
# Distill pendek jadi simpan tiap 200 steps untuk safety
distill_ckpt_overrides = [
    "ckpts.save_per_updates=999999999",  # tidak perlu periodic full save
    "ckpts.last_per_updates=200",        # tapi last checkpoint setiap 200 steps
    "ckpts.keep_last_n_checkpoints=2",   # jaga 2 terakhir
    "ckpts.log_samples=False",
]

# ── Build & run ───────────────────────────────────────────────────────────────
distill_run_name = "F5TTS_EarlyBiMamba_DistillWarmup_datasetku_kaggle_T4_test"

distill_cmd = [
    "uv", "run", "--no-sync", "--python", str(VENV_PY),
    "accelerate", "launch",
    "--num_processes=1",
    f"--mixed_precision={H100_MIXED_PRECISION}",
    # Single-process T4 test: tetap pakai dynamo_backend=no untuk aman
    "--dynamo_backend=no",
    "src/f5_tts/train/train.py",
    "--config-name", "F5TTS_EarlyBiMamba_DistillWarmup.yaml",
    "datasets.name=datasetku",
    f"model.cfm_experiment.teacher_ckpt_path={TEACHER_CKPT}",
    "++model.cfm_experiment.use_distill=true",
    "++model.cfm_experiment.freeze_student_except_mamba_mixers=true",
    f"datasets.batch_size_per_gpu={_distill_batch}",   # ← auto-tuned
    f"datasets.max_samples={_distill_maxsmp}",          # ← fixed T4 test preset
    f"datasets.num_workers={_distill_nwork}",           # ← auto-tuned
    f"optim.epochs={DISTILL_EPOCHS}",
    f"optim.num_warmup_updates={DISTILL_WARMUP}",
    "optim.learning_rate=2e-5",
    "optim.grad_accumulation_steps=1",                  # no grad accum, batch sudah besar
    "model.cfm_experiment.lambda_distill_out=0.0",
    "model.arch.checkpoint_activations=True",           # hemat VRAM untuk T4
    "ckpts.logger=wandb",
    f"ckpts.wandb_project={WANDB_PROJECT}",
    f"ckpts.wandb_run_name={distill_run_name}",
    f"ckpts.save_dir={distill_save_dir_rel}",
    *attn_backend_overrides,
    *distill_ckpt_overrides,                            # ← pakai distill_ckpt, bukan no_periodic
]

run_cmd(distill_cmd, cwd=REPO_DIR, env=h100_env)       # ← h100_env, bukan runtime_env
run_cmd(["ls", "-lah", str(distill_dir_abs)])

# ── Validasi output ───────────────────────────────────────────────────────────
distill_last = distill_dir_abs / "model_last.pt"
if not distill_last.exists():
    raise FileNotFoundError(f"Distill final checkpoint tidak ditemukan: {distill_last}")

print("Distill final checkpoint:", distill_last)
print(f"Distill selesai ({DISTILL_EPOCHS} epochs, ~{DISTILL_EPOCHS * _steps_per_epoch} steps).")
print("Cek wandb: loss_flow harus < 3.0 dan lr harus non-zero sebelum lanjut ke full training.")

In [ ]:
# Jalankan di cell baru — quick sanity infer dari distill checkpoint
infer_script = f"""
import torch
ckpt = torch.load(r'{distill_last}', map_location='cpu', weights_only=False)
keys = list(ckpt.keys())
print("Checkpoint keys:", keys)
if 'step' in ckpt:
    print("Trained steps:", ckpt['step'])
if 'update' in ckpt:
    print("Trained updates:", ckpt['update'])
if 'loss' in ckpt:
    print("Last loss:", ckpt['loss'])
# Cek apakah SSM weights ada dan tidak semua nol
model_sd = ckpt.get('model_state_dict', ckpt.get('ema_model_state_dict', {{}}))
mamba_keys = [k for k in model_sd if 'mamba' in k.lower() or 'ssm' in k.lower() or 'dt_proj' in k.lower()]
print(f"Mamba/SSM keys found: {{len(mamba_keys)}}")
for k in mamba_keys[:5]:
    t = model_sd[k]
    print(f"  {{k}}: shape={{tuple(t.shape)}} mean={{t.float().mean().item():.6f}} std={{t.float().std().item():.6f}}")
"""
run_py(["-c", infer_script], cwd=REPO_DIR)

In [ ]:
# 11) Full training phase (T4 test preset, final checkpoint only)
import subprocess as _sp

distill_last = distill_dir_abs / "model_last.pt"
if not distill_last.exists():
    raise FileNotFoundError(
        f"Distill checkpoint belum ada: {distill_last}. Jalankan cell distill dulu."
    )

# Probe VRAM untuk auto-tune full-phase
vram_probe = _sp.run(
    ["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", "-c",
     "import torch; print(torch.cuda.get_device_properties(0).total_memory // (1024**2))"],
    cwd=str(REPO_DIR), env=runtime_env, capture_output=True, text=True,
)
try:
    _vram_mb = int(vram_probe.stdout.strip().splitlines()[-1])
except Exception:
    _vram_mb = 40960

_full_batch = 9600
_full_maxsmp = 32
_full_nwork = 4

print(f"VRAM detected: {_vram_mb} MB ({_vram_mb // 1024} GB)")
print(f"Auto-tuned full batch : {_full_batch} frames")
print(f"Auto-tuned max_samples: {_full_maxsmp}")
print(f"Auto-tuned num_workers: {_full_nwork}")

full_run_name = "F5TTS_EarlyBiMamba_v1_datasetku_kaggle_T4_test"

attn_backend_overrides = []
if H100_USE_FLASH_ATTN:
    try:
        run_py_nosync(["-c", "import flash_attn; print('flash_attn available')"], cwd=REPO_DIR, timeout=60)
        attn_backend_overrides = ["model.arch.attn_backend=flash_attn"]
        print("Using flash_attn backend.")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("flash_attn tidak tersedia. Fallback ke backend torch.")

full_env = runtime_env.copy()
full_env.update({
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,roundup_power2_divisions:16",
    "CUDA_LAUNCH_BLOCKING": "0",
    "OMP_NUM_THREADS": str(_full_nwork),
    "MKL_NUM_THREADS": str(_full_nwork),
    "TORCH_ALLOW_TF32_CUBLAS_OVERRIDE": "1",
    "TORCH_CUDNN_V8_API_ENABLED": "1",
    "PYTHONFAULTHANDLER": "1",
})

full_cmd = [
    "uv",
    "run",
    "--no-sync",
    "--python",
    str(VENV_PY),
    "accelerate",
    "launch",
    "--num_processes=1",
    f"--mixed_precision={H100_MIXED_PRECISION}",
    "--dynamo_backend=no",
    "src/f5_tts/train/train.py",
    "--config-name",
    "F5TTS_EarlyBiMamba_v1.yaml",
    "datasets.name=datasetku",
    f"++ckpts.pretrained_model_path={distill_last}",
    "++model.cfm_experiment.use_distill=false",
    "++model.cfm_experiment.freeze_student_except_mamba_mixers=false",
    f"datasets.batch_size_per_gpu={_full_batch}",
    f"datasets.max_samples={_full_maxsmp}",
    f"datasets.num_workers={_full_nwork}",
    "optim.epochs=5",
    "optim.num_warmup_updates=500",
    "optim.grad_accumulation_steps=1",
    "model.arch.checkpoint_activations=True",
    "ckpts.logger=wandb",
    f"ckpts.wandb_project={WANDB_PROJECT}",
    f"ckpts.wandb_run_name={full_run_name}",
    f"ckpts.save_dir={full_save_dir_rel}",
    *attn_backend_overrides,
    *no_periodic_ckpt_overrides,
]

run_cmd(full_cmd, cwd=REPO_DIR, env=full_env)
run_cmd(["ls", "-lah", str(full_dir_abs)])

full_last = full_dir_abs / "model_last.pt"
if not full_last.exists():
    raise FileNotFoundError(f"Full-training final checkpoint tidak ditemukan: {full_last}")

print("\nFinal checkpoints:")
print("- Distill final:", distill_last)
print("- Full final:", full_last)

In [ ]:
# 12) Push final checkpoints ke Hugging Face Hub (new private repo)
# Upload dijalankan langsung dari kernel notebook agar prosesnya stabil di Kaggle.
if not distill_last.exists() or not full_last.exists():
    raise FileNotFoundError(
        "Checkpoint final belum lengkap. Jalankan cell full training dulu."
    )

# Install huggingface_hub ke environment notebook (bukan venv) kalau belum ada
try:
    from huggingface_hub import HfApi
except ImportError:
    import subprocess as _sp
    _sp.run(["pip", "install", "-q", "-U", "huggingface_hub"], check=True)
    from huggingface_hub import HfApi

from datetime import datetime, timezone
from uuid import uuid4

api = HfApi(token=HF_TOKEN)

whoami = api.whoami(token=HF_TOKEN)
owner = (HF_OUTPUT_REPO_OWNER or whoami.get("name") or "").strip()
if not owner:
    raise RuntimeError(
        "Tidak bisa menentukan owner Hugging Face. "
        "Isi HF_OUTPUT_REPO_OWNER atau pastikan token HF valid."
    )

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
target_repo_id = None
for attempt in range(5):
    candidate_repo_id = f"{owner}/{HF_OUTPUT_REPO_PREFIX}-{timestamp}-{uuid4().hex[:8]}"
    try:
        api.create_repo(
            repo_id=candidate_repo_id,
            repo_type="model",
            private=True,
            exist_ok=False,
            token=HF_TOKEN,
        )
        target_repo_id = candidate_repo_id
        break
    except Exception as exc:
        msg = str(exc).lower()
        if not any(token in msg for token in ("already exists", "409", "conflict")):
            raise
        print(f"Repo conflict untuk {candidate_repo_id}, retrying...")

if target_repo_id is None:
    raise RuntimeError("Gagal membuat repo HF baru tanpa konflik setelah beberapa percobaan.")

print("Target HF repo:", target_repo_id)
print("Mode: private + always-new repo")

model_card = "\n".join([
    "# KCV-TTS T4 Test Checkpoints",
    "",
    "Repo ini dibuat otomatis dari notebook trainzeh6.1.",
    f"- Distill checkpoint: {distill_last.name}",
    f"- Full checkpoint: {full_last.name}",
    "",
    "Checkpoint disimpan private secara default.",
])
readme_path = WORKDIR / "README_t4_test_upload.md"
readme_path.write_text(model_card, encoding="utf-8")

api.upload_file(
    path_or_fileobj=str(distill_last),
    path_in_repo="checkpoints/distill/model_last.pt",
    repo_id=target_repo_id,
    repo_type="model",
    token=HF_TOKEN,
)
print("✓ Distill checkpoint uploaded")

api.upload_file(
    path_or_fileobj=str(full_last),
    path_in_repo="checkpoints/full/model_last.pt",
    repo_id=target_repo_id,
    repo_type="model",
    token=HF_TOKEN,
)
print("✓ Full checkpoint uploaded")

api.upload_file(
    path_or_fileobj=str(readme_path),
    path_in_repo="README.md",
    repo_id=target_repo_id,
    repo_type="model",
    token=HF_TOKEN,
)
print("✓ README uploaded")

print("\nUpload selesai.")
print("Repo URL:", f"https://huggingface.co/{target_repo_id}")
print("Distill URL:", f"https://huggingface.co/{target_repo_id}/blob/main/checkpoints/distill/model_last.pt")
print("Full URL:", f"https://huggingface.co/{target_repo_id}/blob/main/checkpoints/full/model_last.pt")


In [ ]:
# One-cell inference test untuk trainzeh6.1 (output ke /kaggle/working)
from pathlib import Path
from IPython.display import Audio, display
import csv
import os
import subprocess

INFER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pakai full checkpoint kalau ada, fallback ke distill
ckpt_file = Path(full_last if "full_last" in globals() and Path(full_last).exists() else distill_last)
if not ckpt_file.exists():
    raise FileNotFoundError(f"Checkpoint tidak ditemukan: {ckpt_file}")

# Ambil 1 sample referensi dari metadata hasil training
if not MERGED_CSV.exists():
    raise FileNotFoundError(f"MERGED_CSV tidak ditemukan: {MERGED_CSV}")

with open(MERGED_CSV, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter="|")
    rows = list(reader)

if not rows:
    raise ValueError(f"metadata kosong: {MERGED_CSV}")


def resolve_audio_path(p: str) -> Path:
    p = str(p).strip()
    cand = Path(p)
    candidates = [
        cand,
        DATASET_ROOT / p,
        DATASET_DATA_DIR / p,
        REPO_DIR / p,
        WORKDIR / p,
        INPUT_ROOT / p,
    ]
    for c in candidates:
        if c.exists():
            return c.resolve()
    raise FileNotFoundError(f"Audio referensi tidak ketemu: {p}")


ref_row = next((r for r in rows if str(r.get("audio_file", "")).strip() and str(r.get("text", "")).strip()), None)
if ref_row is None:
    raise ValueError("Tidak ada baris valid di metadata_merged.csv")

ref_audio = resolve_audio_path(ref_row["audio_file"])
ref_text = str(ref_row["text"]).strip()

# Ganti teks ini sesuai kebutuhan
gen_text = "HALO"

# Cari vocab
vocab_file = REPO_DIR / "data" / "Emilia_ZH_EN_pinyin" / "vocab.txt"
if not vocab_file.exists():
    fallback_vocab = next((p for p in (REPO_DIR / "data").glob("**/vocab.txt") if p.is_file() and p.stat().st_size > 0), None)
    if fallback_vocab is None:
        raise FileNotFoundError("vocab.txt tidak ditemukan")
    vocab_file = fallback_vocab

out_wav = INFER_OUTPUT_DIR / "infer_trainzeh6_1_test.wav"

infer_script = f"""
from pathlib import Path
from f5_tts.api import F5TTS
import torch

ckpt_file = Path(r"{str(ckpt_file)}")
vocab_file = Path(r"{str(vocab_file)}")
ref_audio = Path(r"{str(ref_audio)}")
out_wav = Path(r"{str(out_wav)}")
ref_text = {ref_text!r}
gen_text = {gen_text!r}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device   =", device)
print("ckpt     =", ckpt_file)
print("vocab    =", vocab_file)
print("ref_audio=", ref_audio)
print("ref_text =", ref_text)
print("gen_text =", gen_text)

tts = F5TTS(
    model="F5TTS_EarlyBiMamba_v1",
    ckpt_file=str(ckpt_file),
    vocab_file=str(vocab_file),
    use_ema=True,
    device=device,
)

if hasattr(tts, "ema_model") and tts.ema_model is not None:
    tts.ema_model.float()
if hasattr(tts, "model") and tts.model is not None:
    tts.model.float()

tts.infer(
    ref_file=str(ref_audio),
    ref_text=ref_text,
    gen_text=gen_text,
    file_wave=str(out_wav),
    nfe_step=32,
    speed=1.0,
    seed=1234,
)

print("DONE:", out_wav)
"""

infer_env = dict(runtime_env if "runtime_env" in globals() else os.environ)
infer_env["MPLBACKEND"] = "Agg"

if "run_py" in globals():
    run_py(["-c", infer_script], cwd=REPO_DIR, env=infer_env)
else:
    subprocess.run(
        ["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", "-c", infer_script],
        cwd=str(REPO_DIR),
        env=infer_env,
        check=True,
        text=True,
    )

print("Saved:", out_wav)
display(Audio(str(out_wav)))


## Notes
- Notebook ini sudah Kaggle-native: dataset dibaca dari `/kaggle/input`, lalu disalin ke `/kaggle/temp` untuk seluruh proses kerja.
- Repo, venv, dataset kerja, dan checkpoint training berada di `/kaggle/temp`.
- Output inference test disimpan ke `/kaggle/working/inference_tests/infer_trainzeh6_1_test.wav`.
- Kalau OOM, turunkan `datasets.batch_size_per_gpu` bertahap (contoh: `9600 -> 7680 -> 6400`).
- Checkpoint hanya dibuat saat akhir fase distill dan akhir fase full training (tanpa checkpoint periodik).
- Full training tidak dari nol: inisialisasi partial load dari parent checkpoint yang sama dengan teacher distill (`Eempostor/F5-TTS-INDO-FINETUNE-V2`).
- Cell upload HF akan selalu membuat repo model private baru dengan suffix acak agar tidak konflik, lalu upload kedua checkpoint final.
- API key/token bisa diisi hardcode di cell 1 atau fallback ke environment variable dengan nama yang sama.
